# Duplicates in Database
There are 547 peptides in 135 isoforms that match with a canonical peptide. They are in the databse, but they should not be so the fasta was adjusted but the search was not run again due to time constraints. Out of the 135 isoforms, 62 are not uniquely identifiable and 73 are uniquely identifiable, but not with the duplicated peptides.

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import numpy as np
import re
import openpyxl
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/05_MaxQuant"):
    print("WARNING: The working directory is not set to the '05_MaxQuant'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '05_MaxQuant' folder (\"{cwd}\").")

In [ ]:
# Data directories
prenylation_dir = "../04_Prenylation_Database/data/"
mapping_dir = "../01_UniProt/data/mapping/"
isoform_database_dir = "../02_Isoform_Database/"

total_extracts_dir = "data/ProteinGroups_final/te/"
prenylation_enrichment_dir = "data/ProteinGroups_final/clickit/"
peptides_dir = "data/peptides_final/"
removed_peptides_dir = "data/peptides_final/removed_peptides/"

## Read in MaxQuant Results and Lists of isoforms and prenylated proteins

In [ ]:
isoform_fasta = read_fastafile(os.path.join(isoform_database_dir, 'Isoform_Database_only_iso.fasta'))
removed_peptides = pd.read_csv(os.path.join(isoform_database_dir, 'removed_peptides.csv'))

#total extracts
mq_peptides_act_te = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_act_te_LNc_peptides.xlsx')) # Th1-Cells activated vs non-activated
mq_peptides_stat_te = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_stat_te_LNc_peptides.xlsx')) # activated Th1 cells +/- statin treatment
mq_peptides_FTI_te = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_FTI_te_LNc_peptides.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

#prenylation enriched (with clickit-reaction)
mq_peptides_act_clickit = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_act_clickit_LNc_peptides.xlsx')) # Th1-Cells activated vs non-activated
mq_peptides_stat_clickit = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_stat_clickit_LNc_peptides.xlsx')) # activated Th1 cells +/- statin treatment
mq_peptides_FTI_clickit = pd.read_excel(os.path.join(peptides_dir, 'Prenyl_FTI_clickit_LNc_peptides.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

In [ ]:
duplicates_with_canonical = ['P0DPB3-2', 'Q1A5X6-3', 'Q9NPD5-2', 'Q9NQG6-2', 'O14578-4', 'Q59H18-1', 'O14787-2', 'O14815-2', 'O15119-2', 'O43508-2', 
                             'O60333-3', 'Q8IX05-2', 'O75182-2', 'O75638-2', 'O76041-2', 'O95361-2', 'P63092-4', 'O95925-3', 'P01854-1', 'P01871-1', 
                             'P06493-2', 'P09471-2', 'P09651-2', 'P0CAP1-11', 'Q6EEV4-2', 'Q9BZG1-2', 'Q9BZG1-4', 'P0DMS9-2', 'P0DP58-2', 'P35638-2', 
                             'P0DTE4-1', 'P0DTE4-4', 'P10827-2', 'P10912-2', 'P22392-2', 'P19440-3', 'P23416-2', 'P31483-2', 'P35609-2', 'P38405-3', 
                             'P42167-2', 'Q8N726-2', 'P46597-3', 'P46934-4', 'P48059-3', 'P49760-3', 'P50225-2', 'P53671-3', 'P57737-3', 'Q9P2S2-2', 
                             'Q7Z6K5-2', 'P59998-2', 'P67936-2', 'Q00887-2', 'Q03001-3', 'Q06413-6', 'Q08477-2', 'Q13404-1', 'E9PAV3-2', 'Q13950-2', 
                             'Q14999-2', 'Q5SQ64-2', 'Q5VSY0-2', 'Q5VZM2-2', 'Q68DX3-4', 'G9CGD6-2', 'Q6WCQ1-3', 'Q9Y3L3-2', 'Q70YC5-4', 'Q7KZ85-2', 
                             'Q7Z4H7-2', 'Q92843-2', 'Q86XR7-2', 'Q9ULR0-1', 'Q8IVL1-4', 'Q9P2H3-3', 'Q8IXZ2-2', 'Q8N103-3', 'Q8N2Z9-2', 'Q8TD91-2', 
                             'Q92637-2', 'Q92841-3', 'Q96F07-2', 'Q9BXH1-2', 'Q96PK6-5', 'Q96Q89-4', 'Q96S37-2', 'Q99435-2', 'Q9BQS2-3', 'Q9BT81-2', 
                             'Q9BTE6-3', 'Q9GZN2-2', 'Q9GZU2-4', 'Q9Y4C0-3', 'Q9Y4C0-4', 'Q9NQV6-6', 'Q9NQW5-1', 'Q9NR96-4', 'Q9NVL1-3', 'Q9NVV4-2', 
                             'Q9P0J0-2', 'Q9UII6-4', 'Q9UPN3-5', 'O94854-4', 'Q9Y3E7-3', 'Q9Y6Q2-2', 'Q9Y6Q2-3', 'A7E2F4-3', 'O75335-2', 'O75678-3', 
                             'P0DTL6-1', 'Q3SY52-3', 'Q3ZCX4-3', 'Q92617-3', 'Q96HJ9-2', 'Q96HQ0-4', 'Q96I27-2', 'Q9NX45-3', 'Q9UK12-2', 'A6QL64-3', 
                             'B4DH59-2', 'P0C7U1-2', 'Q3KNT7-3', 'Q5TI25-2', 'Q5VTM2-2', 'Q6IEE8-2', 'Q6ZN19-3', 'Q76KX8-2', 'Q96ET8-3', 'Q96MS3-3', 
                             'Q9GZM3-2', 'A6NGB0-1', 'Q8IU53-2', 'Q86UD7-2', 'Q9H1X3-3']

removed_isoforms = ['P0DPB3-2', 'Q1A5X6-3', 'Q59H18-1', 'O14787-2', 'O14815-2', 'O15119-2', 'O75182-2', 'O95361-2', 'O95925-3', 'P01854-1', 'P01871-1', 
                    'P06493-2', 'P09651-2', 'P0CAP1-11', 'Q6EEV4-2', 'P0DMS9-2', 'P35638-2', 'P10912-2', 'P19440-3', 'P23416-2', 'P31483-2', 'P38405-3', 
                    'Q8N726-2', 'P48059-3', 'P49760-3', 'P59998-2', 'Q06413-6', 'Q13950-2', 'Q5SQ64-2', 'Q5VSY0-2', 'Q5VZM2-2', 'Q68DX3-4', 'G9CGD6-2', 
                    'Q6WCQ1-3', 'Q8IVL1-4', 'Q8N103-3', 'Q92841-3', 'Q96F07-2', 'Q96PK6-5', 'Q96S37-2', 'Q99435-2', 'Q9BT81-2', 'Q9BTE6-3', 'Q9NR96-4', 
                    'Q9NVL1-3', 'Q9UPN3-5', 'Q9Y6Q2-2', 'A7E2F4-3', 'O75335-2', 'P0DTL6-1', 'Q3SY52-3', 'Q92617-3', 'Q96HJ9-2', 'Q96HQ0-4', 'Q9NX45-3', 
                    'B4DH59-2', 'Q3KNT7-3', 'Q5TI25-2', 'Q76KX8-2', 'A6NGB0-1', 'Q8IU53-2', 'Q9H1X3-3']

In [ ]:
removed_peptides_set = set(removed_peptides['Captured_Peptide'])

In [ ]:
def filter_removed_peptides(df, removed_peptides_set,
                             id_col='Sequence'):
    """
    Cleans MaxQuant results and keeps only the rows where the sequence column
    matches a peptide inside the removed_peptides_set.
    """
    # 1. Basic Cleaning (Decoys, Contaminants)
    initial_count = len(df)
    clean_df = df.copy()
    
    # Remove rows based on MaxQuant flag columns
    flags = ['Reverse', 'Potential contaminant']
    for col in flags:
        if col in clean_df.columns:
            clean_df = clean_df[clean_df[col] != '+']
    
    # Filter by Unique Peptides threshold
    if 'Unique peptides' in clean_df.columns:
        clean_df = clean_df[clean_df['Unique peptides'] >= min_unique_peptides]
    
    # 2. Filter rows where the Sequence is in your removed_peptides_set
    # Direct vectorized lookup using .isin()
    df_removed_peptides = clean_df[clean_df[id_col].isin(removed_peptides_set)].copy()
    
    # 3. Reporting
    print(f"--- Filter Report ---")
    print(f"Initial Rows: {initial_count}")
    print(f"Cleaned Rows: {len(clean_df)}")
    print(f"Matching Removed Peptides Rows: {len(df_removed_peptides)}")
    
    return df_removed_peptides

In [ ]:
# Execution:
#total extracts
removed_act_te = filter_removed_peptides(mq_peptides_act_te, removed_peptides_set)
removed_stat_te = filter_removed_peptides(mq_peptides_stat_te, removed_peptides_set)
removed_FTI_te = filter_removed_peptides(mq_peptides_FTI_te, removed_peptides_set)

#prentylation enrichment
removed_act_clickit = filter_removed_peptides(mq_peptides_act_clickit, removed_peptides_set)
removed_stat_clickit = filter_removed_peptides(mq_peptides_stat_clickit, removed_peptides_set)
removed_FTI_clickit = filter_removed_peptides(mq_peptides_FTI_clickit, removed_peptides_set)

In [ ]:
# Save as csv 
removed_act_te.to_csv(os.path.join(removed_peptides_dir, 'removed_act_te.csv'), index=False)
removed_stat_te.to_csv(os.path.join(removed_peptides_dir, 'removed_stat_te.csv'), index=False)
removed_FTI_te.to_csv(os.path.join(removed_peptides_dir, 'removed_FTI_te.csv'), index=False)

removed_act_clickit.to_csv(os.path.join(removed_peptides_dir, 'removed_act_clickit.csv'), index=False)
removed_stat_clickit.to_csv(os.path.join(removed_peptides_dir, 'removed_stat_clickit.csv'), index=False)
removed_FTI_clickit.to_csv(os.path.join(removed_peptides_dir, 'removed_FTI_clickit.csv'), index=False)